In [0]:
import pyspark.sql.functions as F



In [0]:
input_path = 's3://ims-dbr-raw/fhvhv_trip_records/'
output_path = 's3://ims-dbr-stage/fhvhv_trip_records/'

In [0]:
df = spark.read.parquet(input_path)
display(df)

#Schema

In [0]:
df_stage = df.withColumn('hvfhs_license_num', F.col('hvfhs_license_num').cast('string')) \
.withColumn('dispatching_base_num', F.col('dispatching_base_num').cast('string')) \
.withColumn('originating_base_num', F.col('originating_base_num').cast('string')) \
.withColumn('request_datetime', F.col('request_datetime').cast('timestamp')) \
.withColumn('on_scene_datetime', F.col('on_scene_datetime').cast('timestamp')) \
.withColumn('pickup_datetime', F.col('pickup_datetime').cast('timestamp')) \
.withColumn('dropoff_datetime', F.col('dropoff_datetime').cast('timestamp')) \
.withColumn('PULocationID', F.col('PULocationID').cast('int')) \
.withColumn('DOLocationID', F.col('DOLocationID').cast('int')) \
.withColumn('trip_miles', F.col('trip_miles').cast('double')) \
.withColumn('trip_time', F.col('trip_time').cast('int')) \
.withColumn('base_passenger_fare', F.col('base_passenger_fare').cast('double')) \
.withColumn('tolls', F.col('tolls').cast('double')) \
.withColumn('bcf', F.col('bcf').cast('double')) \
.withColumn('sales_tax', F.col('sales_tax').cast('double')) \
.withColumn('congestion_surcharge', F.col('congestion_surcharge').cast('double')) \
.withColumn('airport_fee', F.col('airport_fee').cast('double')) \
.withColumn('tips', F.col('tips').cast('double')) \
.withColumn('driver_pay', F.col('driver_pay').cast('double')) \
.withColumn('shared_request_flag', F.col('shared_request_flag').cast('String')) \
.withColumn('shared_match_flag', F.col('shared_match_flag').cast('String')) \
.withColumn('access_a_ride_flag', F.col('access_a_ride_flag').cast('String')) \
.withColumn('wav_request_flag', F.col('wav_request_flag').cast('String')) \
.withColumn('wav_match_flag', F.col('wav_match_flag').cast('String')) \
.withColumn('trip_duration', F.col('trip_time')/60) \
    \
.withColumn('pu_hour_of_day', F.hour('pickup_datetime')) \
.withColumn('pu_day_of_week', F.dayofweek('pickup_datetime')) \
.withColumn('pu_year', F.year('pickup_datetime')) \
.withColumn('pu_month', F.month('pickup_datetime')) \
.withColumn('do_hour_of_day', F.hour('dropoff_datetime')) \
.withColumn('do_day_of_week', F.dayofweek('dropoff_datetime')) \
.withColumn('do_year', F.year('dropoff_datetime')) \
.withColumn('do_month', F.month('dropoff_datetime'))\
    \
.withColumn('is_pu_unknown', (F.col('PULocationID') == 264) | (F.col('PULocationID') == 265)) \
.withColumn('is_do_unknown', (F.col('DOLocationID') == 264) | (F.col('DOLocationID') == 265)) \
.withColumn('is_originating_base_unknown', F.col('originating_base_num').isNull()) \
.withColumn('is_on_scene_datetime_missing', F.col('on_scene_datetime').isNull()) \
.withColumn('is_do_unknown', (F.col('DOLocationID') == 264) | (F.col('DOLocationID') == 265)) \
.withColumn('is_pu_unknown', (F.col('PULocationID') == 264) | (F.col('PULocationID') == 265)) \
.filter(
    (F.col('PULocationID') <= 265) &
    (F.col('DOLocationID') <= 265) &
    (F.col('base_passenger_fare') > 0) &
    (F.col('driver_pay') > 0) &
    (F.col('trip_duration') > 0) &
    (F.col('trip_miles') > 0)) \
.dropDuplicates(subset = ['dispatching_base_num', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID'])

#Writing to Silver

In [0]:
## Write to Stage Table
df_stage.write \
  .mode("overwrite") \
  .parquet(output_path)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nytaxi.staging.fhvhv_trip_records
USING PARQUET
LOCATION 's3://ims-dbr-stage/fhvhv_trip_records/';